# Fine-Tuning do Qwen3-4B com Unsloth (LoRA / QLoRA)

**Professor:** Renan Santos Mendes

**Contato:** renansantosmendes@gmail.com

---

Este notebook apresenta um pipeline completo de fine-tuning supervisionado
(SFT) do modelo Qwen3-4B utilizando a biblioteca Unsloth, com adaptadores
LoRA em precisao 4-bit (QLoRA). O material esta dividido em duas partes:

1. **Treinamento**: preparacao do ambiente, configuracao, carregamento do
   modelo base, aplicacao do LoRA, treinamento supervisionado e publicacao
   do modelo no Hugging Face Hub.
2. **Inferencia**: carregamento do modelo treinado e execucao de um chat
   interativo no terminal.


## Parte 1 - Treinamento do modelo

### 1.1 Preparacao do ambiente

Nesta secao sao clonados os arquivos auxiliares do curso (dataset de
exemplo) e instaladas as bibliotecas necessarias para o treinamento.


A celula abaixo clona o repositorio do curso, que contem o dataset
utilizado no treinamento. O comando `%%capture` e usado para suprimir a
saida do `git clone` no notebook.


In [ ]:
%%capture
!git clone https://github.com/renansantosmendes/ai_runtime_core.git


A celula abaixo instala a biblioteca `unsloth` e sua dependencia
`psutil`. Conforme padrao da disciplina, o gerenciador de pacotes `uv` e
utilizado para a instalacao no Google Colab.


In [ ]:
!uv pip install unsloth psutil -qqq

A celula abaixo importa a classe `FastLanguageModel` do Unsloth e
desabilita avisos de `FutureWarning`, que nao sao relevantes para a
execucao deste material.


In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

from unsloth import FastLanguageModel

print("Unsloth carregado com sucesso.")


### 1.2 Configuracao do treinamento

A celula abaixo importa as bibliotecas restantes utilizadas ao longo do
treinamento e define a classe `Config`, que centraliza todos os
hiperparametros e caminhos utilizados no pipeline. Concentrar essas
informacoes em um unico lugar facilita a reproducao e o ajuste do
experimento.


In [ ]:
import os
from typing import Any, Optional

import torch
from unsloth import FastLanguageModel
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset, Dataset
from transformers import TextStreamer
from trl import SFTTrainer, SFTConfig
import matplotlib.pyplot as plt


class Config:
    """Central configuration for the fine-tuning run.

    This class groups model, dataset, LoRA and training
    hyperparameters in a single place, making the experiment
    easier to reproduce and adjust.
    """

    MODEL_NAME = "unsloth/Qwen3-4B"
    MAX_SEQ_LENGTH = 2048
    LOAD_IN_4BIT = True

    DATA_PATH = (
        "/content/ai_runtime_core/genai_and_advanced_analytics/datasets/"
    )
    DATA_FILE = "synapseai_knowledge_base.jsonl"
    USE_EVAL_SPLIT = True
    EVAL_SIZE = 0.10
    ENABLE_THINKING = False

    LORA_RANK = 16
    LORA_ALPHA = 16
    LORA_DROPOUT = 0
    USE_RSLORA = True
    TARGET_MODULES = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]

    BATCH_SIZE = 2
    GRAD_ACCUM = 4
    EPOCHS = 1
    LEARNING_RATE = 1e-4
    WARMUP_RATIO = 0.05
    WEIGHT_DECAY = 0.01
    MAX_GRAD_NORM = 1.0
    LR_SCHEDULER = "cosine"
    LOGGING_STEPS = 5
    EVAL_STEPS = 20
    SEED = 3407
    OUTPUT_DIR = "outputs"

    PUSH_TO_HUB = True
    HF_USERNAME = "renansantosmendes"
    MODEL_REPO = "SynapseAI-Qwen3-4B-Instruct"
    DATASET_REPO = "SynapseAI-Knowledge-Base"


### 1.3 Prompt de sistema padrao

A celula abaixo define o prompt de sistema padrao utilizado tanto durante
o treinamento (para contextualizar o formato de conversa) quanto durante a
inferencia. Ele descreve a persona do assistente virtual que sera
treinado.


In [ ]:
DEFAULT_SYSTEM_PROMPT = (
    "Voce e um assistente virtual especializado da SynapseAI Solutions. "
    "A SynapseAI e uma empresa que desenvolve produtos corporativos "
    "baseados em Inteligencia Artificial Generativa para os setores de "
    "Financas, Educacao e Saude.\\n\\n"
    "Seus produtos incluem:\\n"
    "- FinBrain: Copiloto financeiro corporativo\\n"
    "- RiskGen: Plataforma de risco, compliance e auditoria\\n"
    "- EduMentor AI: Tutor educacional inteligente\\n"
    "- CourseGen Studio: Geracao de conteudo educacional\\n"
    "- ClinicaGPT: Assistente clinico de apoio a decisao\\n\\n"
    "Responda de forma clara, precisa e profissional sobre a empresa, "
    "seus produtos, tecnologias e servicos."
)


### 1.4 Funcoes auxiliares

As proximas celulas definem as funcoes auxiliares utilizadas pelo pipeline
de treinamento: formatacao do dataset no template de chat, visualizacao da
curva de perda, geracao de respostas com streaming, obtencao do token do
Hugging Face e publicacao dos artefatos no Hub.


A funcao `format_dataset` abaixo aplica o template de chat do Qwen3 a
cada exemplo do dataset, convertendo a lista de mensagens (coluna
`messages`) em uma unica string de texto pronta para o treinamento.


In [ ]:
def format_dataset(
    example: dict,
    tokenizer: Any,
    config: Config,
) -> dict:
    """Apply the Qwen3 chat template to a single dataset example.

    Args:
        example: A single dataset row containing a 'messages' key
            with the chat-formatted conversation (list of dicts
            with 'role' and 'content' keys).
        tokenizer: The tokenizer associated with the base model,
            used to render the chat template.
        config: The training configuration, used to read whether
            the Qwen3 'thinking' mode should be enabled.

    Returns:
        A dictionary with a single 'text' key containing the
        rendered chat string.
    """
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=config.ENABLE_THINKING,
    )
    return {"text": text}


A funcao `plot_loss` abaixo percorre o historico de logs do `trainer` e
plota as curvas de perda de treinamento e de validacao, salvando o
resultado em um arquivo de imagem.


In [ ]:
def plot_loss(trainer: Any) -> None:
    """Plot the training and evaluation loss curves.

    Reads the trainer's log history, extracts the training and
    (when available) evaluation loss values per step, plots both
    curves and saves the resulting figure as 'training_loss.png'.

    Args:
        trainer: A trained Hugging Face / TRL trainer instance
            exposing a 'state.log_history' attribute.

    Returns:
        None. The plot is displayed and saved to disk as a side
        effect.
    """
    log_history = trainer.state.log_history

    train_steps, train_losses = [], []
    eval_steps, eval_losses = [], []

    for log in log_history:
        if "loss" in log and "step" in log:
            train_steps.append(log["step"])
            train_losses.append(log["loss"])
        if "eval_loss" in log and "step" in log:
            eval_steps.append(log["step"])
            eval_losses.append(log["eval_loss"])

    plt.figure(figsize=(10, 6))
    plt.plot(
        train_steps,
        train_losses,
        label="Training Loss",
        color="blue",
        marker="o",
        markersize=4,
    )
    if eval_losses:
        plt.plot(
            eval_steps,
            eval_losses,
            label="Eval Loss",
            color="red",
            marker="s",
            markersize=4,
        )
    plt.xlabel("Steps (Passos de Treinamento)")
    plt.ylabel("Loss (Perda)")
    plt.title("Curva de Convergencia")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.savefig("training_loss.png", dpi=120)
    plt.show()
    print("Grafico salvo como 'training_loss.png'")


A funcao `predict_response` abaixo coloca o modelo em modo de inferencia
e gera uma resposta em tempo real (streaming) para uma pergunta fornecida,
utilizando o prompt de sistema padrao caso nenhum outro seja informado.


In [ ]:
def predict_response(
    model: Any,
    tokenizer: Any,
    config: Config,
    question: str,
    system_prompt: Optional[str] = None,
    max_new_tokens: int = 1024,
) -> None:
    """Generate and stream a model response for a given question.

    Switches the model to inference mode, builds the chat prompt
    using the system and user messages, and streams the generated
    tokens directly to standard output.

    Args:
        model: The fine-tuned language model.
        tokenizer: The tokenizer associated with the model.
        config: The training configuration, used to read whether
            the Qwen3 'thinking' mode should be enabled.
        question: The user question to send to the model.
        system_prompt: An optional system prompt overriding the
            default persona. Defaults to None, in which case
            'DEFAULT_SYSTEM_PROMPT' is used.
        max_new_tokens: The maximum number of new tokens to
            generate. Defaults to 1024.

    Returns:
        None. The generated answer is streamed to standard output
        as a side effect.

    Example:
        >>> predict_response(
        ...     model,
        ...     tokenizer,
        ...     config,
        ...     "Quais os produtos da SynapseAI?",
        ... )
    """
    FastLanguageModel.for_inference(model)

    if system_prompt is None:
        system_prompt = DEFAULT_SYSTEM_PROMPT

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=config.ENABLE_THINKING,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True)

    print(f"\\nUsuario: {question}")
    print("Assistente: ", end="")

    _ = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        temperature=0.1,
        use_cache=True,
    )
    print("\\n" + "-" * 50)


A funcao `get_hf_token` abaixo obtem o token de autenticacao do Hugging
Face a partir dos segredos do Google Colab ou, alternativamente, de uma
variavel de ambiente, permitindo a execucao tanto no Colab quanto em outros
ambientes.


In [ ]:
def get_hf_token() -> str:
    """Fetch the Hugging Face authentication token.

    Attempts to read the token from Google Colab secrets first,
    falling back to the 'HF_TOKEN' environment variable when the
    Colab API is not available.

    Returns:
        The Hugging Face authentication token as a string.

    Raises:
        ValueError: If the token cannot be found in either the
            Colab secrets or the environment variables.
    """
    try:
        from google.colab import userdata
        return userdata.get("HF_TOKEN")
    except Exception:
        token = os.environ.get("HF_TOKEN")
        if not token:
            raise ValueError(
                "Token nao encontrado. Configure o HF_TOKEN nos "
                "segredos do Colab ou exporte a variavel de "
                "ambiente HF_TOKEN."
            )
        return token


A funcao `push_to_hub` abaixo publica os adaptadores LoRA, o tokenizer e
o dataset formatado no Hugging Face Hub, alem de salvar uma copia local do
dataset em formato JSONL.


In [ ]:
def push_to_hub(
    model: Any,
    tokenizer: Any,
    dataset: Dataset,
    config: Config,
) -> None:
    """Push the LoRA adapters, tokenizer and dataset to the Hub.

    Args:
        model: The fine-tuned language model whose LoRA adapters
            will be pushed to the Hugging Face Hub.
        tokenizer: The tokenizer associated with the model.
        dataset: The formatted dataset to be pushed to the Hub
            and saved locally as a JSONL file.
        config: The training configuration, used to read the
            Hugging Face username and repository names.

    Returns:
        None. The model, tokenizer and dataset are uploaded, and
        a local copy of the dataset is written to disk, as side
        effects.
    """
    token = get_hf_token()

    model_repo_id = f"{config.HF_USERNAME}/{config.MODEL_REPO}"
    print(f"Enviando modelo para: {model_repo_id} ...")
    model.push_to_hub(model_repo_id, token=token)
    tokenizer.push_to_hub(model_repo_id, token=token)
    print(f"Modelo disponivel em: https://huggingface.co/{model_repo_id}")

    dataset_repo_id = f"{config.HF_USERNAME}/{config.DATASET_REPO}"
    print(f"Enviando dataset para: {dataset_repo_id} ...")
    dataset.push_to_hub(dataset_repo_id, token=token)
    print(
        "Dataset disponivel em: "
        f"https://huggingface.co/datasets/{dataset_repo_id}"
    )

    local_filename = "synapseai_dataset_formatado.jsonl"
    dataset.to_json(local_filename, force_ascii=False)
    print(f"Dataset salvo localmente como '{local_filename}'")


### 1.5 Pipeline principal de treinamento

A funcao `main` abaixo orquestra o pipeline completo: carregamento do
dataset, carregamento do modelo base, aplicacao do LoRA, formatacao do
dataset, configuracao e execucao do `SFTTrainer`, visualizacao da curva de
perda, geracao de exemplos de resposta e, por fim, publicacao dos
artefatos no Hugging Face Hub.


In [ ]:
def main() -> None:
    """Run the full fine-tuning pipeline.

    Loads the dataset and the base model, applies LoRA adapters,
    formats the dataset with the chat template, trains the model
    with the 'SFTTrainer', plots the resulting loss curves,
    generates a couple of sample responses and, when enabled,
    pushes the resulting artifacts to the Hugging Face Hub.

    Returns:
        None.
    """
    config = Config()

    print("Carregando dataset...")
    dataset = load_dataset(
        path=config.DATA_PATH,
        data_files=config.DATA_FILE,
        split="train",
    )
    print(f"{len(dataset)} exemplos carregados.")

    print("Carregando modelo e tokenizer...")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=config.MODEL_NAME,
        max_seq_length=config.MAX_SEQ_LENGTH,
        load_in_4bit=config.LOAD_IN_4BIT,
        use_gradient_checkpointing="unsloth",
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Aplicando LoRA...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=config.LORA_RANK,
        target_modules=config.TARGET_MODULES,
        lora_alpha=config.LORA_ALPHA,
        lora_dropout=config.LORA_DROPOUT,
        bias="none",
        use_rslora=config.USE_RSLORA,
        use_gradient_checkpointing="unsloth",
        random_state=config.SEED,
    )

    print("Formatando dataset com o chat template...")
    dataset = dataset.map(
        lambda example: format_dataset(example, tokenizer, config)
    )

    eval_dataset = None
    if config.USE_EVAL_SPLIT:
        split = dataset.train_test_split(
            test_size=config.EVAL_SIZE,
            seed=config.SEED,
        )
        train_dataset, eval_dataset = split["train"], split["test"]
        print(
            f"Treino: {len(train_dataset)} | "
            f"Validacao: {len(eval_dataset)}"
        )
    else:
        train_dataset = dataset

    print("\\n----- Exemplo formatado -----")
    print(train_dataset[0]["text"][:1000])
    print("-----------------------------\\n")

    print("Configurando o SFTTrainer...")
    sft_args = dict(
        dataset_text_field="text",
        per_device_train_batch_size=config.BATCH_SIZE,
        gradient_accumulation_steps=config.GRAD_ACCUM,
        warmup_ratio=config.WARMUP_RATIO,
        num_train_epochs=config.EPOCHS,
        learning_rate=config.LEARNING_RATE,
        logging_steps=config.LOGGING_STEPS,
        optim="adamw_8bit",
        weight_decay=config.WEIGHT_DECAY,
        max_grad_norm=config.MAX_GRAD_NORM,
        lr_scheduler_type=config.LR_SCHEDULER,
        seed=config.SEED,
        output_dir=config.OUTPUT_DIR,
        report_to="none",
    )
    if eval_dataset is not None:
        sft_args.update(
            eval_strategy="steps",
            eval_steps=config.EVAL_STEPS,
            per_device_eval_batch_size=config.BATCH_SIZE,
        )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=SFTConfig(**sft_args),
    )
    trainer = train_on_responses_only(
        trainer,
        instruction_part="<|im_start|>user\n",
        response_part="<|im_start|>assistant\n",
    )

    print("\\nIniciando o treinamento...\\n")
    trainer.train()
    print("\\nTreinamento concluido.\\n")

    plot_loss(trainer)

    predict_response(
        model,
        tokenizer,
        config,
        "quais os produtos da synapse ai?",
    )
    predict_response(
        model,
        tokenizer,
        config,
        "a empresa synapse tem alguma solucao para industria?",
    )

    if config.PUSH_TO_HUB:
        push_to_hub(model, tokenizer, dataset, config)

    print("\\nPronto.")


A celula abaixo executa o pipeline de treinamento completo definido na
funcao `main`.


In [ ]:
if __name__ == "__main__":
    main()


## Parte 2 - Inferencia e chat interativo

Apos o treinamento, esta secao apresenta como carregar o modelo publicado
no Hugging Face Hub e utiliza-lo em um chat interativo pelo terminal.


A celula abaixo importa as bibliotecas necessarias para a inferencia e
define as constantes utilizadas: o identificador do modelo publicado no
Hub, o tamanho maximo de sequencia, o modo de quantizacao e o prompt de
sistema padrao (repetido aqui para que esta secao possa ser executada de
forma independente da Parte 1).


In [ ]:
import os
from typing import Any, Optional

from unsloth import FastLanguageModel
from transformers import TextStreamer


MODEL_ID = "renansantosmendes/SynapseAI-Qwen3-4B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True
ENABLE_THINKING = False

DEFAULT_SYSTEM_PROMPT = (
    "Voce e um assistente virtual especializado da SynapseAI Solutions. "
    "A SynapseAI e uma empresa que desenvolve produtos corporativos "
    "baseados em Inteligencia Artificial Generativa para os setores de "
    "Financas, Educacao e Saude.\\n\\n"
    "Seus produtos incluem:\\n"
    "- FinBrain: Copiloto financeiro corporativo\\n"
    "- RiskGen: Plataforma de risco, compliance e auditoria\\n"
    "- EduMentor AI: Tutor educacional inteligente\\n"
    "- CourseGen Studio: Geracao de conteudo educacional\\n"
    "- ClinicaGPT: Assistente clinico de apoio a decisao\\n\\n"
    "Responda de forma clara, precisa e profissional sobre a empresa, "
    "seus produtos, tecnologias e servicos."
)


A funcao `load_model` abaixo carrega o modelo base com os adaptadores
LoRA treinados aplicados, ja preparando-o para o modo de inferencia.


In [ ]:
def load_model(model_id: str = MODEL_ID) -> tuple:
    """Load the base model with the fine-tuned LoRA adapters.

    Args:
        model_id: The Hugging Face Hub repository identifier of
            the fine-tuned model. Defaults to 'MODEL_ID'.

    Returns:
        A tuple '(model, tokenizer)' with the loaded model, ready
        for inference, and its associated tokenizer.
    """
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_id,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=LOAD_IN_4BIT,
        token=os.environ.get("HF_TOKEN"),
    )
    FastLanguageModel.for_inference(model)
    model.generation_config.max_length = None
    return model, tokenizer


A funcao `chat` abaixo gera e transmite (streaming) a resposta do modelo
para uma unica pergunta do usuario, utilizando o prompt de sistema padrao
caso nenhum outro seja informado.


In [ ]:
def chat(
    model: Any,
    tokenizer: Any,
    question: str,
    system_prompt: Optional[str] = None,
    max_new_tokens: int = 1024,
) -> None:
    """Generate and stream a single chat answer.

    Args:
        model: The fine-tuned language model, already prepared
            for inference.
        tokenizer: The tokenizer associated with the model.
        question: The user question to send to the model.
        system_prompt: An optional system prompt overriding the
            default persona. Defaults to None, in which case
            'DEFAULT_SYSTEM_PROMPT' is used.
        max_new_tokens: The maximum number of new tokens to
            generate. Defaults to 1024.

    Returns:
        None. The generated answer is streamed to standard output
        as a side effect.
    """
    if system_prompt is None:
        system_prompt = DEFAULT_SYSTEM_PROMPT

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": question},
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=ENABLE_THINKING,
    )
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    streamer = TextStreamer(tokenizer, skip_prompt=True)

    print("Assistente: ", end="")
    _ = model.generate(
        **inputs,
        streamer=streamer,
        max_new_tokens=max_new_tokens,
        temperature=0.1,
        use_cache=True,
    )
    print()


A funcao `main` abaixo carrega o modelo treinado e executa um loop de
chat interativo no terminal, encerrando quando o usuario digitar um dos
comandos de saida (`sair`, `exit`, `quit` ou `q`).


In [ ]:
def main() -> None:
    """Run an interactive chat loop in the terminal.

    Loads the fine-tuned model and repeatedly prompts the user
    for a question, streaming the model's answer until an exit
    command is entered.

    Returns:
        None.
    """
    print("Carregando o modelo...")
    model, tokenizer = load_model()
    print("Modelo carregado. Digite 'sair' para encerrar.\\n")

    while True:
        question = input("Voce: ").strip()
        if question.lower() in {"sair", "exit", "quit", "q"}:
            print("Ate logo!")
            break
        if not question:
            continue
        chat(model, tokenizer, question)
        print("-" * 50)


A celula abaixo inicia o chat interativo definido na funcao `main`.


In [ ]:
if __name__ == "__main__":
    main()
